In [1]:
import os
import re
import glob
import json
import numpy as np
import pandas as pd
import torch

INPUT = "/kaggle/input/datasets/nocturnalnerd18"
SRC = f"{INPUT}/patch-qwen-paths"
HELD = ["val", "test"]
N_BOOT = 1000
MIN_MEAN = 0.05

with open(f"{SRC}/path_meta.json") as f:
    pmeta = json.load(f)
SENDER_NAMES = pmeta["sender_names"]
SET_NAMES = pmeta["set_names"]
DIRECTIONS = pmeta["directions"]
PARTS = [n[len("heads_"):] for n in SET_NAMES if n.startswith("heads_") and n != "heads_all"]
k = {n: i for i, n in enumerate(SET_NAMES)}

In [2]:
def batch_number(path):
    return int(re.search(r"_batch_(\d+)\.pt$", path).group(1))

In [3]:
entries = []
for path in sorted(glob.glob(f"{SRC}/P_batch_*.pt"), key=batch_number):
    entries.extend(torch.load(path))
good = [e for e in entries if e["ok"]]
print(f"{len(entries)} entries, {len(good)} ok;", pd.Series([f"{e['hallu_type']}/{e['split']}/{e['stratum']}" for e in good]).value_counts().sort_index().to_string())

E = np.stack([e["effects"].numpy() for e in good])[:, 0]
sign = np.array([e["sign"] for e in good])
role = np.array([e["stratum"] for e in good])
uid = np.array([e["uid"] for e in good])
htype = np.array([e["hallu_type"] for e in good])
split = np.array([e["split"] for e in good])
SUP = -sign[:, None, None] * E

386 entries, 386 ok; attribute/test/false_yes     10
attribute/test/true_yes      10
attribute/train/false_yes    73
attribute/train/true_yes     73
attribute/val/false_yes      10
attribute/val/true_yes       10
relation/test/false_yes      15
relation/test/true_yes       15
relation/train/false_yes     70
relation/train/true_yes      70
relation/val/false_yes       15
relation/val/true_yes        15


In [4]:
def routes(S):
    g = lambda name: S[..., k[name]]
    out = {"total": g("total"), "direct": g("direct"), "heads_only": g("heads_all") - g("direct")}
    for part in PARTS:
        out[f"mlp_{part}"] = g(f"heads_{part}") - g("heads_all")
    out["interaction"] = g("total") - g("heads_all") - sum(out[f"mlp_{part}"] for part in PARTS)
    return out

In [5]:
R = routes(SUP)
COLS = ["total", "heads_only"] + [f"mlp_{x}" for x in PARTS] + ["interaction"]
rng = np.random.default_rng(1)

In [6]:
def contrast(sel):
    fy = np.flatnonzero((role == "false_yes") & sel)
    ct = np.flatnonzero((role == "true_yes") & sel)
    fy_ix = {u: i for u, i in zip(uid[fy], fy)}
    ct_ix = {u: i for u, i in zip(uid[ct], ct)}
    pairs = sorted(set(fy_ix) & set(ct_ix))
    if len(pairs) < 8:
        return None
    a = np.array([fy_ix[u] for u in pairs])
    b = np.array([ct_ix[u] for u in pairs])
    boot = rng.integers(0, len(pairs), size=(N_BOOT, len(pairs)))
    res = {}
    for c in COLS:
        diff = R[c][a] - R[c][b]
        means = diff[boot].mean(1)
        res[c] = (diff.mean(0), np.percentile(means, 2.5, axis=0), np.percentile(means, 97.5, axis=0))
    return len(pairs), res

In [7]:
summary = {}
for ht in ["attribute", "relation"]:
    train = contrast((htype == ht) & (split == "train"))
    held = contrast((htype == ht) & np.isin(split, HELD))
    if train is None or held is None:
        print(f"{ht}: too few pairs for a split check")
        continue
    print(f"===== {ht}: train {train[0]} pairs, held-out (val+test) {held[0]} pairs; false_yes minus true_yes, mean [95% interval] =====")
    rows, agree = [], []
    for i, name in enumerate(SENDER_NAMES):
        for c in COLS:
            mt, lt, ut = (x[i] for x in train[1][c])
            mh, lh, uh = (x[i] for x in held[1][c])
            if abs(mt) >= MIN_MEAN and (lt > 0 or ut < 0):
                rows.append({"sender": name, "route": c, "train": f"{mt:+.3f} [{lt:+.3f}, {ut:+.3f}]", "held_out": f"{mh:+.3f} [{lh:+.3f}, {uh:+.3f}]",
                             "same_sign": bool(np.sign(mt) == np.sign(mh)), "held_excludes_0": bool(lh > 0 or uh < 0)})
                summary[f"{ht}/{name}/{c}"] = {"train": float(mt), "held": float(mh)}
    tab = pd.DataFrame(rows)
    print(tab.to_string(index=False) if len(tab) else "no route clears the train threshold")
    if len(tab):
        print(f"{ht}: {int(tab.same_sign.sum())} of {len(tab)} train-significant routes keep their sign held out; {int(tab.held_excludes_0.sum())} also exclude 0 on only {held[0]} held-out pairs")

with open("/kaggle/working/path_split_check.json", "w") as f:
    json.dump(summary, f)

===== attribute: train 73 pairs, held-out (val+test) 20 pairs; false_yes minus true_yes, mean [95% interval] =====
       sender       route                   train                held_out  same_sign  held_excludes_0
 L15 H26 text       total +0.233 [+0.134, +0.341] +0.110 [-0.042, +0.280]       True            False
 L15 H26 text interaction +0.163 [+0.077, +0.244] +0.077 [-0.044, +0.209]       True            False
  L15 MLP all       total +0.861 [+0.424, +1.304] +0.773 [-0.079, +1.567]       True            False
  L15 MLP all  heads_only +0.056 [+0.023, +0.092] +0.091 [+0.030, +0.148]       True             True
  L15 MLP all   mlp_19_21 +0.120 [+0.017, +0.230] +0.172 [-0.112, +0.512]       True            False
  L15 MLP all interaction +0.631 [+0.221, +1.011] +0.496 [-0.312, +1.231]       True            False
   L20 H2 all       total +0.306 [+0.121, +0.455] +0.344 [+0.046, +0.664]       True             True
   L20 H2 all   mlp_22_27 +0.130 [+0.057, +0.191] +0.166 [+0.041, +0.